# Tutorial 7: Multi-Armed Bandits & Thompson Sampling

This notebook covers the explore-vs-exploit problem in recommendation: epsilon-greedy, Upper Confidence Bound (UCB), and Thompson Sampling.

**The gap this fills:** our ALS and Neural CF models (Tutorials 3-4) are trained once on historical data and never update. Bandits continuously learn from every interaction instead.

**The simulation framing (read this first):** Instacart is a static, historical dataset — there's no real "online" stream of user visits we can experiment on. To teach bandits meaningfully with this data, we simulate a simplified online setting:

- **Arms** = a subset of products (the top ~50 by popularity). Each arm has a fixed, hidden "true" purchase probability — the fraction of users who actually bought that product.
- **Rounds** = users arriving one at a time. At each round, the bandit picks ONE product (arm) to "show" to the arriving user.
- **Reward** = 1 if that user's actual purchase history contains the shown product, 0 otherwise. This comes straight from our real interaction matrix, so it's grounded in genuine purchase behavior — even though "one product per round" is a simplification of a real recommender (which shows many items at once).

This mirrors the classic bandit problem (e.g. picking which ad or product to feature) and lets us compute real regret and exposure metrics using data we already have from earlier tutorials. It also reuses `measure_item_exposure()` / `calculate_gini_coefficient()`-style fairness thinking from Tutorial 6.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle

from utils import load_processed_data

plt.style.use('seaborn-v0_8-darkgrid')


## Step 1: Load Data

Same preprocessed data as every other tutorial in this series — `train_matrix`, `test_matrix`, and `product_info`. No new data prep needed for bandits.


In [ ]:
processed_data_path = 'data/processed'

data = load_processed_data(processed_data_path)

train_matrix = data['train_matrix']
test_matrix = data['test_matrix']
product_info = data['product_info']

print(f"Products: {train_matrix.shape[1]:,}")
print(f"Users: {train_matrix.shape[0]:,}")


## Step 2: Select Bandit Arms

We restrict to a manageable number of popular products as "arms." Real Instacart purchase probabilities are tiny (roughly 0.1%-3% for popular items) — with too many arms, algorithms like UCB need far more rounds than practical to statistically separate such small probability gaps.

**50 arms** keeps the problem tractable for a video-length demo while still being a meaningful multi-way choice.


In [ ]:
def select_bandit_arms(train_matrix, product_info, n_arms=50):
    """
    Select a subset of products to act as "arms" for the bandit simulation.

    We restrict to popular products because very rare products would have
    near-zero purchase probability, making it hard to see meaningful
    exploration/exploitation dynamics in a reasonably-sized simulation.

    Parameters:
    -----------
    train_matrix : scipy sparse matrix (users x items)
        Used to rank products by popularity (number of purchases)

    product_info : DataFrame
        Product metadata, used to attach names/departments to arms

    n_arms : int, default=50
        Number of top products to use as bandit arms

    Returns:
    --------
    arm_item_indices : numpy array
        Product indices (into the original catalog) chosen as arms.
        arm i corresponds to product arm_item_indices[i].
    """

    print(f"\nSelecting top {n_arms} products as bandit arms...")

    # Popularity = number of purchases per product (sum down each column)
    item_popularity = np.asarray(train_matrix.sum(axis=0)).flatten()

    # Take the n_arms most popular product indices
    arm_item_indices = np.argsort(item_popularity)[::-1][:n_arms]

    print(f"  Selected {len(arm_item_indices)} arms")
    print(f"  Most popular arm has {item_popularity[arm_item_indices[0]]:.0f} purchases")
    print(f"  Least popular selected arm has {item_popularity[arm_item_indices[-1]]:.0f} purchases")

    return arm_item_indices


In [ ]:
N_ARMS = 50

arm_item_indices = select_bandit_arms(
    train_matrix=train_matrix,
    product_info=product_info,
    n_arms=N_ARMS
)


## Step 3: Ground-Truth Arm Probabilities

**Important:** we use `test_matrix` here, NOT `train_matrix`. These probabilities represent genuinely held-out purchase behavior — the bandit itself never sees this directly, it only observes one reward per round, just like a real bandit would. We only use this for computing regret afterward, to check how well each strategy actually learned.

We also define `get_reward()` here — this simulates "showing" a product to a user and observing whether they actually bought it (a Bernoulli draw governed by the item's true, hidden-to-the-bandit purchase probability).


In [ ]:
def get_true_arm_probabilities(interaction_matrix, arm_item_indices):
    """
    Compute the TRUE (hidden-from-the-bandit) purchase probability for each arm.

    This is only used for evaluation (e.g. computing regret) - the bandit
    algorithms themselves never see this directly, they have to learn it
    through trial and error, exactly like a real bandit would.

    Parameters:
    -----------
    interaction_matrix : scipy sparse matrix (users x items)
        Ground truth purchases (we use test_matrix so this reflects
        genuinely held-out behavior, not what the bandit already trained on)

    arm_item_indices : numpy array
        Product indices selected as arms (from select_bandit_arms)

    Returns:
    --------
    true_probs : numpy array, shape (n_arms,)
        true_probs[i] = fraction of users who purchased arm i's product
    """

    n_users = interaction_matrix.shape[0]

    arm_purchases = np.asarray(
        interaction_matrix[:, arm_item_indices].sum(axis=0)
    ).flatten()

    true_probs = arm_purchases / n_users

    return true_probs


def get_reward(user_idx, arm_item_idx, interaction_matrix):
    """
    Simulate showing `arm_item_idx` to `user_idx` and observing a reward.

    Reward = 1 if the user's real purchase history contains this product,
    0 otherwise.

    Parameters:
    -----------
    user_idx : int
        The user "arriving" this round

    arm_item_idx : int
        Product index (in the original catalog) that was shown

    interaction_matrix : scipy sparse matrix (users x items)
        Ground truth interactions used to check if a purchase occurred

    Returns:
    --------
    reward : int (0 or 1)
    """

    return int(interaction_matrix[user_idx, arm_item_idx] > 0)


In [ ]:
true_arm_probs = get_true_arm_probabilities(
    interaction_matrix=test_matrix,
    arm_item_indices=arm_item_indices
)

print("True purchase probabilities (held-out test data):")
print(f"  Best arm probability: {true_arm_probs.max():.4f}")
print(f"  Worst arm probability: {true_arm_probs.min():.4f}")
print(f"  Average arm probability: {true_arm_probs.mean():.4f}")

best_arm_product_idx = arm_item_indices[np.argmax(true_arm_probs)]
best_arm_row = product_info[product_info['product_idx'] == best_arm_product_idx]
if not best_arm_row.empty:
    print(f"  Best arm product: {best_arm_row['product_name'].values[0]}")


## Step 4: The Three Bandit Strategies

Now the core algorithms. Each one answers the same question differently: **which arm do we pull this round, given what we've learned so far?**

- **Epsilon-greedy**: dead simple, easy to explain, but wastes exploration budget on clearly-bad arms forever.
- **UCB**: exploration is *optimistic* — arms we're uncertain about get a bonus, but the bonus shrinks the more we pull that arm.
- **Thompson Sampling**: fully Bayesian — we keep a belief distribution (Beta) over each arm's true probability, and sample from it to decide. Naturally balances explore/exploit without a tunable epsilon.


### Epsilon-Greedy

With probability `epsilon`: explore (pick a uniformly random arm). Otherwise: exploit (pick the arm with the best observed average reward so far, among arms tried at least once).

**A bug worth knowing about (fixed here):** an earlier version of this function gave untried arms an optimistic default average, which caused the exploit branch to silently sweep through every arm once and lock onto whichever arm got lucky with an early success — producing good-looking short-term numbers without genuinely confirming it found the best arm. Untried arms should only get picked through the random explore branch — that's the standard textbook definition, and what's implemented below.


In [ ]:
def epsilon_greedy_select_arm(arm_reward_sums, arm_pull_counts, epsilon=0.1):
    """
    Choose an arm using the epsilon-greedy strategy.

    Parameters:
    -----------
    arm_reward_sums : numpy array
        Running total reward observed for each arm so far

    arm_pull_counts : numpy array
        Number of times each arm has been pulled so far

    epsilon : float, default=0.1
        Probability of exploring (random arm) instead of exploiting.

    Returns:
    --------
    chosen_arm : int
        Index of the arm to pull this round
    """

    n_arms = len(arm_pull_counts)

    # Explore: random arm, uniform probability
    if np.random.random() < epsilon:
        return np.random.randint(n_arms)

    # Exploit: pick arm with highest average reward among TRIED arms only.
    # Untried arms get -inf so they can never be picked here - they can
    # only enter the picture through the random explore branch above.
    with np.errstate(divide='ignore', invalid='ignore'):
        avg_rewards = np.where(
            arm_pull_counts > 0,
            arm_reward_sums / np.maximum(arm_pull_counts, 1),
            -np.inf
        )

    # Edge case: very first round, nothing tried yet - fall back to random
    if np.all(arm_pull_counts == 0):
        return np.random.randint(n_arms)

    return int(np.argmax(avg_rewards))


### Upper Confidence Bound (UCB)

UCB is "optimistic in the face of uncertainty" — it adds a confidence bonus to each arm's average reward:

$$\text{score}(i) = \text{avg\_reward}(i) + c \times \sqrt{\frac{2 \ln(\text{total\_rounds})}{\text{pulls}(i)}}$$

Arms pulled fewer times get a bigger bonus (we're less sure about them), so they naturally get explored more. As pulls grow, the bonus shrinks and the algorithm settles into exploiting the best arm.

**Tuning note:** `c=1.0` reproduces the textbook UCB1 formula. But at real Instacart purchase-probability scales (roughly 0.1%-3%, very tightly clustered), even `c=1.0` is too conservative — the confidence bonus stays larger than the actual probability gaps between arms for a very long time, so UCB barely explores past a random baseline within a practical round count. We use **`c=0.15`** here, trading away some of UCB's theoretical worst-case guarantee for practical convergence speed at this probability scale. This is a real, well-documented characteristic of UCB (its regret bound is asymptotic) — not a bug to code around, and worth discussing on camera.


In [ ]:
def ucb_select_arm(arm_reward_sums, arm_pull_counts, total_rounds, c=0.15):
    """
    Choose an arm using the UCB1 strategy.

    Parameters:
    -----------
    arm_reward_sums : numpy array
        Running total reward for each arm

    arm_pull_counts : numpy array
        Number of times each arm has been pulled

    total_rounds : int
        Total number of rounds played so far (used inside the log term)

    c : float, default=0.15
        Exploration strength multiplier on top of the standard UCB1 bonus.
        (Textbook default is 1.0 - see note above on why we use 0.15 here.)

    Returns:
    --------
    chosen_arm : int
        Index of the arm to pull this round
    """

    # Make sure every arm is tried at least once first - UCB's confidence
    # bonus is undefined (division by zero) for an untried arm
    untried = np.where(arm_pull_counts == 0)[0]
    if len(untried) > 0:
        return int(untried[0])

    avg_rewards = arm_reward_sums / arm_pull_counts

    # Confidence bonus - bigger for arms we've pulled less
    confidence_bonus = c * np.sqrt(2 * np.log(total_rounds) / arm_pull_counts)

    ucb_scores = avg_rewards + confidence_bonus

    return int(np.argmax(ucb_scores))


### Thompson Sampling

For each arm, we maintain a **Beta(alpha, beta)** distribution representing our current belief about that arm's true purchase probability:
- `alpha` = 1 + number of successes (purchases) observed
- `beta` = 1 + number of failures (no purchase) observed

At each round, we draw one random sample from EACH arm's belief distribution, and pick the arm with the highest sample. Arms we're uncertain about (wide Beta distribution) occasionally sample high, naturally driving exploration. Arms we're confident are good (narrow, high Beta distribution) will consistently sample high too.

No epsilon or `c` to tune here — the explore/exploit balance falls out naturally from the uncertainty in each arm's belief distribution.


In [ ]:
def thompson_sampling_select_arm(arm_alpha, arm_beta):
    """
    Choose an arm using Thompson Sampling with a Beta-Bernoulli model.

    Parameters:
    -----------
    arm_alpha : numpy array
        Alpha parameter (successes + 1) for each arm's Beta distribution

    arm_beta : numpy array
        Beta parameter (failures + 1) for each arm's Beta distribution

    Returns:
    --------
    chosen_arm : int
        Index of the arm to pull this round
    """

    # Sample one value from each arm's current belief distribution
    sampled_values = np.random.beta(arm_alpha, arm_beta)

    return int(np.argmax(sampled_values))


## Step 5: The Simulation Loop

This is where it all comes together. At each round, a user "arrives" (sampled at random), the bandit picks an arm using whichever strategy we're testing, we observe the reward via `get_reward()`, and the bandit updates its internal state (pull counts, reward sums, and for Thompson Sampling, its Beta parameters).

`random` is included as a fourth "strategy" — pure random choice, no learning at all. It's a sanity-check baseline: every real bandit strategy should convincingly beat it, the same way we checked ALS/Neural CF against a random baseline in earlier tutorials.


In [ ]:
def run_bandit_simulation(interaction_matrix, arm_item_indices, strategy,
                          n_rounds=100000, epsilon=0.1, ucb_c=0.15,
                          random_state=42):
    """
    Run a full bandit simulation for a given strategy.

    Parameters:
    -----------
    interaction_matrix : scipy sparse matrix (users x items)
        Ground truth purchases used to generate rewards

    arm_item_indices : numpy array
        Product indices selected as arms (from select_bandit_arms)

    strategy : str
        One of 'random', 'epsilon_greedy', 'ucb', 'thompson_sampling'

    n_rounds : int, default=100000
        Number of user arrivals to simulate

    epsilon : float, default=0.1
        Only used when strategy='epsilon_greedy'

    ucb_c : float, default=0.15
        Only used when strategy='ucb'

    random_state : int, default=42
        For reproducibility

    Returns:
    --------
    history : dict with:
        - arms_pulled: list of arm indices chosen, one per round
        - rewards: list of rewards observed, one per round
        - arm_pull_counts: final pull count per arm
        - arm_reward_sums: final total reward per arm
    """

    np.random.seed(random_state)

    n_arms = len(arm_item_indices)
    n_users = interaction_matrix.shape[0]

    # Bandit's internal state - what it has learned so far
    arm_pull_counts = np.zeros(n_arms, dtype=int)
    arm_reward_sums = np.zeros(n_arms, dtype=float)

    # Thompson Sampling needs its own belief parameters (Beta distribution)
    # Start with alpha=beta=1, i.e. a uniform "no idea yet" prior
    arm_alpha = np.ones(n_arms, dtype=float)
    arm_beta = np.ones(n_arms, dtype=float)

    arms_pulled = []
    rewards = []

    print(f"\nRunning {strategy} simulation for {n_rounds:,} rounds...")

    for round_num in range(1, n_rounds + 1):

        # Simulate a random user "arriving" this round
        user_idx = np.random.randint(n_users)

        # Pick an arm according to the chosen strategy
        if strategy == 'random':
            # Sanity-check baseline - no learning at all
            arm_idx = np.random.randint(n_arms)
        elif strategy == 'epsilon_greedy':
            arm_idx = epsilon_greedy_select_arm(arm_reward_sums, arm_pull_counts, epsilon)
        elif strategy == 'ucb':
            arm_idx = ucb_select_arm(arm_reward_sums, arm_pull_counts, round_num, ucb_c)
        elif strategy == 'thompson_sampling':
            arm_idx = thompson_sampling_select_arm(arm_alpha, arm_beta)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        # "Show" the product and observe whether the user actually bought it
        product_idx = arm_item_indices[arm_idx]
        reward = get_reward(user_idx, product_idx, interaction_matrix)

        # Update bandit state with what we just observed
        arm_pull_counts[arm_idx] += 1
        arm_reward_sums[arm_idx] += reward

        if strategy == 'thompson_sampling':
            if reward == 1:
                arm_alpha[arm_idx] += 1
            else:
                arm_beta[arm_idx] += 1

        arms_pulled.append(arm_idx)
        rewards.append(reward)

    print(f"  Total reward collected: {sum(rewards):,} / {n_rounds:,} rounds")
    print(f"  Overall reward rate: {sum(rewards)/n_rounds:.4f}")

    return {
        'strategy': strategy,
        'arms_pulled': arms_pulled,
        'rewards': rewards,
        'arm_pull_counts': arm_pull_counts,
        'arm_reward_sums': arm_reward_sums,
    }


## Step 6: Regret & Running All Strategies

**Regret** at each round = (best possible reward rate) - (reward rate of the arm we actually chose). Cumulative regret is the running total — how much reward we "left on the table" by not always pulling the best arm from the very start.

A good bandit strategy has cumulative regret that grows fast early on (while exploring) and then flattens out (once it's found and is exploiting the best arm).

`compare_bandit_strategies()` runs all four strategies (random, epsilon-greedy, UCB, Thompson Sampling) under identical conditions — same seed, so the same simulated user arrivals — for a fair comparison.


In [ ]:
def calculate_cumulative_regret(history, true_arm_probs):
    """
    Calculate cumulative regret over the course of a simulation.

    Parameters:
    -----------
    history : dict
        Output from run_bandit_simulation()

    true_arm_probs : numpy array
        True purchase probability for each arm - only used here for
        evaluation, never seen by the bandit itself

    Returns:
    --------
    cumulative_regret : numpy array, shape (n_rounds,)
        Running total regret after each round
    """

    best_possible_prob = true_arm_probs.max()

    per_round_regret = np.array([
        best_possible_prob - true_arm_probs[arm_idx]
        for arm_idx in history['arms_pulled']
    ])

    cumulative_regret = np.cumsum(per_round_regret)

    return cumulative_regret


def compare_bandit_strategies(interaction_matrix, arm_item_indices,
                              true_arm_probs, n_rounds=100000,
                              epsilon=0.1, ucb_c=0.15, random_state=42):
    """
    Run all four strategies under identical conditions and collect results
    for comparison (regret curves, final reward rates, exposure).

    Returns:
    --------
    results : dict
        {strategy_name: {'history': ..., 'cumulative_regret': ...}}
    """

    # 'random' is included as a sanity-check baseline - a strategy with zero
    # learning. All three real bandit strategies should beat it convincingly.
    strategies = ['random', 'epsilon_greedy', 'ucb', 'thompson_sampling']
    results = {}

    for strategy in strategies:
        history = run_bandit_simulation(
            interaction_matrix=interaction_matrix,
            arm_item_indices=arm_item_indices,
            strategy=strategy,
            n_rounds=n_rounds,
            epsilon=epsilon,
            ucb_c=ucb_c,
            random_state=random_state  # same seed -> same simulated users across strategies
        )

        cumulative_regret = calculate_cumulative_regret(history, true_arm_probs)

        results[strategy] = {
            'history': history,
            'cumulative_regret': cumulative_regret,
            'final_regret': cumulative_regret[-1],
            'total_reward': sum(history['rewards']),
        }

    return results


In [ ]:
# 100,000 rounds is enough for strategies to meaningfully separate at real
# purchase-probability scales. Feel free to bump this up - your workstation
# handled 20,000 rounds in a few seconds, so 100K+ should still be fast.
N_ROUNDS = 100000

comparison_results = compare_bandit_strategies(
    interaction_matrix=test_matrix,   # reward comes from held-out purchases
    arm_item_indices=arm_item_indices,
    true_arm_probs=true_arm_probs,
    n_rounds=N_ROUNDS,
    epsilon=0.1,
    ucb_c=0.15,
    random_state=42
)


## Step 7: Strategy Comparison Table

Quick numeric comparison: total reward, reward rate, and final regret for each strategy. We also do a sanity check — the best real strategy should convincingly beat random.


In [ ]:
print(f"{'Strategy':<20}{'Total Reward':<15}{'Reward Rate':<15}{'Final Regret':<15}")
print("-" * 65)
for strategy, result in comparison_results.items():
    total_reward = result['total_reward']
    reward_rate = total_reward / N_ROUNDS
    final_regret = result['final_regret']
    print(f"{strategy:<20}{total_reward:<15,}{reward_rate:<15.4f}{final_regret:<15.2f}")

random_reward = comparison_results['random']['total_reward']
best_strategy = max(
    (s for s in comparison_results if s != 'random'),
    key=lambda s: comparison_results[s]['total_reward']
)
best_reward = comparison_results[best_strategy]['total_reward']
improvement = (best_reward - random_reward) / random_reward * 100

print(f"\nBest strategy ({best_strategy}) beats random by {improvement:.1f}%")


## Step 8: Measuring Exposure Fairness (Gini Coefficient)

This ties directly back to Tutorial 6's fairness discussion. `get_exposure_from_bandit_history()` converts a bandit's arm-pull history into the same exposure-count format used there, and `calculate_gini_coefficient()` is the identical function from Tutorial 6 — same metric, same interpretation thresholds.

**Worth calling out on camera:** bandits that converge hard onto one best arm concentrate almost ALL exposure onto that single product — often a *worse* (higher) Gini than a static top-10 list, since here we only show one product per round. Pure exploitation is great for reward, but this is a real cost.


In [ ]:
def get_exposure_from_bandit_history(history, arm_item_indices, n_items):
    """
    Convert a bandit's arm-pull history into the same exposure_counts format
    used by measure_item_exposure() in Tutorial 6 - so we can directly reuse
    calculate_gini_coefficient() and the Lorenz curve plotting functions.

    Parameters:
    -----------
    history : dict
        Output from run_bandit_simulation()

    arm_item_indices : numpy array
        Product indices selected as arms (maps arm index -> catalog product index)

    n_items : int
        Total catalog size, so the returned array lines up with
        product_info / other tutorials' indexing

    Returns:
    --------
    exposure_counts : numpy array, shape (n_items,)
    """

    exposure_counts = np.zeros(n_items, dtype=int)

    for arm_idx in history['arms_pulled']:
        product_idx = arm_item_indices[arm_idx]
        exposure_counts[product_idx] += 1

    return exposure_counts


def calculate_gini_coefficient(exposure_counts):
    """
    Calculate Gini coefficient to measure exposure inequality.
    (Same function as Tutorial 6 - 0 = perfectly equal, 1 = maximally unequal.)
    """

    counts = exposure_counts[exposure_counts > 0]

    if len(counts) == 0:
        return 1.0

    sorted_counts = np.sort(counts)
    n = len(sorted_counts)

    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * sorted_counts)) / (n * np.sum(sorted_counts)) - (n + 1) / n

    return gini


In [ ]:
n_items = train_matrix.shape[1]
gini_by_strategy = {}
exposure_by_strategy = {}

for strategy, result in comparison_results.items():
    exposure_counts = get_exposure_from_bandit_history(
        history=result['history'],
        arm_item_indices=arm_item_indices,
        n_items=n_items
    )
    gini = calculate_gini_coefficient(exposure_counts)

    exposure_by_strategy[strategy] = exposure_counts
    gini_by_strategy[strategy] = gini

    items_exposed = np.sum(exposure_counts > 0)
    print(f"  {strategy:<20} Gini = {gini:.3f}  |  {items_exposed}/{n_items} products ever shown")


## Step 9: What Did Each Strategy Converge On?

For each strategy, find its most-pulled arm and check its true purchase probability against the actual best arm we computed in Step 3. This is the real test of whether a strategy "worked" — not just its reward numbers, but whether it actually found the right answer.


In [ ]:
for strategy, result in comparison_results.items():
    if strategy == 'random':
        continue  # random doesn't "converge" on anything meaningful

    pull_counts = result['history']['arm_pull_counts']
    most_pulled_arm = np.argmax(pull_counts)
    most_pulled_product_idx = arm_item_indices[most_pulled_arm]

    prod_row = product_info[product_info['product_idx'] == most_pulled_product_idx]
    product_name = prod_row['product_name'].values[0] if not prod_row.empty else 'Unknown'

    pct_of_rounds = pull_counts[most_pulled_arm] / N_ROUNDS * 100

    print(f"\n{strategy}:")
    print(f"  Most-pulled arm: {product_name}")
    print(f"  Pulled {pull_counts[most_pulled_arm]:,} times ({pct_of_rounds:.1f}% of all rounds)")
    print(f"  True purchase probability: {true_arm_probs[most_pulled_arm]:.4f}")
    print(f"  (Actual best arm's true probability: {true_arm_probs.max():.4f})")


## Step 10: Visualizations

Three plots:
1. **Cumulative regret** — the classic bandit learning curve. Random is a straight line (no learning); the bandits bend away and flatten as they converge.
2. **Reward rate over time** — a rolling average showing each strategy's reward rate climbing toward the best-possible ceiling.
3. **Exposure fairness** — Gini coefficient bar chart, the explore/exploit-vs-fairness trade-off.


In [ ]:
def plot_cumulative_regret(comparison_results, save_path=None):
    """
    Plot cumulative regret over time for each strategy - the classic bandit
    learning curve. A flattening curve means the strategy has converged.
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {
        'random': 'gray',
        'epsilon_greedy': 'coral',
        'ucb': 'mediumpurple',
        'thompson_sampling': 'skyblue',
    }

    for strategy, result in comparison_results.items():
        ax.plot(
            result['cumulative_regret'],
            label=strategy.replace('_', ' ').title(),
            color=colors.get(strategy, None),
            linewidth=2
        )

    ax.set_xlabel('Round', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cumulative Regret', fontsize=12, fontweight='bold')
    ax.set_title('Cumulative Regret: Lower and Flatter is Better', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
Path('results').mkdir(exist_ok=True)

plot_cumulative_regret(comparison_results, save_path='results/bandit_cumulative_regret.png')


In [ ]:
def plot_reward_rate_over_time(comparison_results, true_arm_probs, window=200,
                               save_path=None):
    """
    Plot the rolling-average reward rate over time for each strategy,
    against the best possible reward rate (a horizontal reference line).
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {
        'random': 'gray',
        'epsilon_greedy': 'coral',
        'ucb': 'mediumpurple',
        'thompson_sampling': 'skyblue',
    }

    for strategy, result in comparison_results.items():
        rewards = np.array(result['history']['rewards'])

        if len(rewards) >= window:
            rolling_avg = np.convolve(rewards, np.ones(window) / window, mode='valid')
            ax.plot(
                np.arange(window - 1, len(rewards)),
                rolling_avg,
                label=strategy.replace('_', ' ').title(),
                color=colors.get(strategy, None),
                linewidth=2
            )

    best_possible = true_arm_probs.max()
    ax.axhline(best_possible, color='black', linestyle='--', linewidth=1.5,
              label=f'Best Possible ({best_possible:.3f})')

    ax.set_xlabel('Round', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'Reward Rate (rolling {window}-round average)', fontsize=12, fontweight='bold')
    ax.set_title('Reward Rate Over Time: Converging Toward the Best Arm', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
plot_reward_rate_over_time(comparison_results, true_arm_probs, window=200,
                           save_path='results/bandit_reward_rate.png')


In [ ]:
def plot_exposure_fairness_comparison(gini_by_strategy, save_path=None):
    """
    Bar chart comparing exposure inequality (Gini coefficient) across
    strategies.
    """

    fig, ax = plt.subplots(figsize=(9, 6))

    strategies = list(gini_by_strategy.keys())
    gini_values = list(gini_by_strategy.values())

    colors_map = {
        'random': 'gray',
        'epsilon_greedy': 'coral',
        'ucb': 'mediumpurple',
        'thompson_sampling': 'skyblue',
    }
    bar_colors = [colors_map.get(s, 'lightgray') for s in strategies]

    bars = ax.bar(
        [s.replace('_', ' ').title() for s in strategies],
        gini_values,
        color=bar_colors,
        alpha=0.8,
        edgecolor='black'
    )

    for bar, value in zip(bars, gini_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height,
               f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

    ax.set_ylabel('Gini Coefficient (Exposure Inequality)', fontsize=12, fontweight='bold')
    ax.set_title('Exposure Fairness by Strategy', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
plot_exposure_fairness_comparison(gini_by_strategy, save_path='results/bandit_exposure_fairness.png')


## Step 11: Save Results

Save everything for reference or reuse — comparison results, true arm probabilities, arm indices, and Gini scores.


In [ ]:
with open('results/bandit_results.pkl', 'wb') as f:
    pickle.dump({
        'comparison_results': comparison_results,
        'true_arm_probs': true_arm_probs,
        'arm_item_indices': arm_item_indices,
        'gini_by_strategy': gini_by_strategy,
    }, f)

print("Saved: results/bandit_results.pkl")


## Summary

**Key findings to check once you run this:**
- Does the best strategy convincingly beat random? (Sanity check — if not, something's wrong.)
- Do epsilon-greedy, UCB, and Thompson Sampling all converge on the *same* best arm? (They should — that's the real test of correctness, not just reward numbers.)
- How does the regret ranking compare to the fairness (Gini) ranking? Expect an inverse relationship: whichever strategy exploits hardest (lowest regret) will also have the worst (highest) exposure fairness.

**Limitations:**
- This is a simplified single-arm-per-round simulation, not a full recommendation slate. Combining bandits with a top-K list (contextual bandits) is a natural next step.
- Reward is based on historical purchase presence, not a true causal "would this recommendation cause a purchase" signal.

**Next steps:**
- Try contextual bandits (arm choice depends on user features)
- Combine bandit exploration with the MMR diversity layer from Tutorial 6
- Experiment with different `epsilon` and `ucb_c` values, and more rounds
